In [55]:
import edsnlp
from sklearn.metrics import precision_recall_fscore_support

In [72]:
nlp = edsnlp.load("../artifacts/model-last", exclude=["doc_classifier_syn"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8718.05it/s]


In [73]:
print(nlp.pipeline)

[('doc_classifier', TrainableDocClassifier(
  (embedding): DocPooler(
    (embedding): Transformer(
      (transformer): CamembertModel(
        (embeddings): CamembertEmbeddings(
          (word_embeddings): Embedding(32005, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
        )
        (encoder): CamembertEncoder(
          (layer): ModuleList(
            (0-11): 12 x CamembertLayer(
              (attention): CamembertAttention(
                (self): CamembertSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=F

In [74]:
text = "Compte rendu d'hospitalisation du service de Chirurgie orthopédique\n\nMadame Gérard Danard a été opérée en ambulatoire dans le service le 24/07/2025 (cf compte-rendu opératoire) pour libération de doigt à ressaut du 4ème rayon de la main gauche.\n\nL'intervention s'est déroulée sous anesthésie locorégionale.\n\nL'intervention s'est déroulée sans problème particulier.\n\nLes suites opératoires ont été simples et Madame Danard a pu sortir à domicile le jour même avec comme traitement de sortie :\n\n- Antalgiques\n\n- Soins locaux tous les 2 jours jusqu'à cicatrisation complète\n\n- Fils résorbables\n\n- Mobilisation du poignet et des doigts sans contre-indication hormis de garder le pansement au propre et au sec.\n\nJe reverrai Madame Danard dans 6 semaines pour un nouveau contrôle clinique.\n\nDr Yves Hadjas .\n"
doc = nlp(text)

In [75]:
doc._.dp

'S610'

In [76]:
label_attr= ["dp"]
val_data = (
  edsnlp.data.read_parquet(
      "/data/scratch/ldedieu/CU2/encoder_baseline/PARHAF",
      #"/data/scratch/ldedieu/CU2/encoder_baseline/MISTRAL/test_v1",
      tokenizer=nlp.tokenizer,
      converter="omop",
      doc_attributes={
          "dp": "dp_gold",
      },
      shuffle="fragment"
  )
)

In [77]:
val_data

Stream(
  reader=ParquetReader(path=['/data/scratch/ldedieu/CU2/encoder_baseline/PARHAF'], shuffle=fragment, loop=False),
  ops=[
    map(<edsnlp.data.converters.OmopDict2DocConverter object at 0x7fd4c2d8cb90>)
  ],
  writer=None)

In [78]:
val_data = val_data.map_pipeline(
    nlp,
)
val_data = val_data.set_processing(num_cpu_workers=8, show_progress=True)

note_nlp = val_data.to_pandas(
    converter="omop",
    doc_attributes=[
        "dp_gold",
        "dp",
    ]
)
note_nlp

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (626 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (986 > 512). Running this sequence through the model will result in indexing errors
2it [00:02,  1.18s/it][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (536 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (724 > 512). Running this sequence through the model will result in indexing errors
50it [00:13,  3.90it/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (931 > 512). Running this sequence through the model w

,note_id,note_text,dp_gold,dp,entities
0,4346506903553,Compte rendu opératoire de Chirurgie orthopédi...,S430,M1901,[]
1,12223476924417,Compte rendu opératoire de Chirurgie viscérale...,Z452,N871,[]
2,11510512353282,Compte rendu d'hospitalisation du service de C...,K409,K409,[]
3,2499670966272,Compte rendu de consultation de Chirurgie orth...,M7954,M7954,[]
4,412316860418,Compte rendu d'hospitalisation du service de C...,G561,S670,[]
...,...,...,...,...,...
856,3874060500992,Compte rendu de consultation de Chirurgie orth...,M6534,M181,[]
857,5188320493570,Compte rendu d'hospitalisation du service de C...,M233,Z530,[]
858,1425929142272,Compte rendu de consultation de Chirurgie orth...,M2537,M2537,[]
859,5781025980416,Compte rendu de consultation de Chirurgie orth...,M2351,S835,[]


In [79]:
df = note_nlp.copy()

In [80]:
def compute_metric(df, granularity=None):
    df_clean = df.dropna(subset=['dp', 'dp_gold'])

    y_true = df_clean['dp_gold']
    y_pred = df_clean['dp']
    
    # Slice the first three characters of each string
    if granularity:
        y_true = df_clean['dp_gold'].str[:granularity]
        y_pred = df_clean['dp'].str[:granularity]

    # Compute Micro metrics
    micro_p, micro_r, micro_f, _ = precision_recall_fscore_support(
        y_true, 
        y_pred, 
        average='micro'
    )

    # Compute Macro metrics 
    # (zero_division=0 prevents warnings if a class in 'dp' is never predicted)
    macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(
        y_true, 
        y_pred, 
        average='macro', 
        zero_division=0
    )

    # Display results
    print(f"Micro - Precision: {micro_p:.4f}, Recall: {micro_r:.4f}, F1: {micro_f:.4f}")
    print(f"Macro - Precision: {macro_p:.4f}, Recall: {macro_r:.4f}, F1: {macro_f:.4f}")

In [81]:
df_filter = {
    "CRC": note_nlp[note_nlp["note_text"].str.contains("Compte rendu de consultation", na=False)],
    "CRH": note_nlp[note_nlp["note_text"].str.contains("Compte rendu d'hospitalisation", na=False)],
    "CRO": note_nlp[note_nlp["note_text"].str.contains("Compte rendu opératoire", na=False)],
    "all": note_nlp
}

for mode in [ "all"]:#"CRC", "CRH", "CRO", "all"]:
    print(f"mode: {mode}")
    compute_metric(df_filter[mode])

mode: all
Micro - Precision: 0.2520, Recall: 0.2520, F1: 0.2520
Macro - Precision: 0.1349, Recall: 0.0817, F1: 0.0921


In [82]:
for mode in [ "all"]:#"CRC", "CRH", "CRO",
    print(f"mode: {mode}")
    compute_metric(df_filter[mode], granularity=-1)

mode: all
Micro - Precision: 0.3740, Recall: 0.3740, F1: 0.3740
Macro - Precision: 0.1836, Recall: 0.1183, F1: 0.1317


# Grouped document prediction

In [91]:
import pandas as pd
df = note_nlp.copy()
df["note_id"] = df["note_id"].astype(str).str[:-2]
df = df.sort_values("note_id")
df

,note_id,note_text,dp_gold,dp,entities
548,100158637342,Compte rendu de consultation de Chirurgie visc...,K811,K800,[]
683,100158637342,Compte rendu opératoire de Chirurgie viscérale...,K811,K800,[]
734,100158637342,Compte rendu d'hospitalisation du service de C...,K811,Z530,[]
126,100759932764,Compte rendu de consultation de Chirurgie visc...,K801,K801,[]
330,100759932764,Compte rendu opératoire de Chirurgie viscérale...,K801,K811,[]
...,...,...,...,...,...
607,98956046499,Compte rendu opératoire de Chirurgie viscérale...,K810,K811,[]
813,98956046499,Compte rendu de consultation de Chirurgie visc...,K810,K800,[]
214,99557341921,Compte rendu de consultation de Chirurgie visc...,K649,K649,[]
377,99557341921,Compte rendu opératoire de Chirurgie viscérale...,K649,K649,[]


In [92]:
df["note_id"].value_counts()

note_id
100158637342    3
100759932764    3
10136122818     3
101361228185    3
101962523607    3
               ..
97152160235     3
97753455656     3
98354751078     3
98956046499     3
99557341921     3
Name: count, Length: 287, dtype: int64

In [93]:
df_grouped = df.groupby("note_id")[["note_text", "dp_gold"]].agg(" ".join).reset_index()
#print(df_grouped["note_text"].values[0])

In [94]:
df_grouped

,note_id,note_text,dp_gold
0,100158637342,Compte rendu de consultation de Chirurgie visc...,K811 K811 K811
1,100759932764,Compte rendu de consultation de Chirurgie visc...,K801 K801 K801
2,10136122818,Compte rendu d'hospitalisation du service de C...,M205 M205 M205
3,101361228185,Compte rendu opératoire de Chirurgie viscérale...,K429 K429 K429
4,101962523607,Compte rendu de consultation de Chirurgie visc...,K801 K801 K801
...,...,...,...
282,97152160235,Compte rendu d'hospitalisation du service de C...,K602 K602 K602
283,97753455656,Compte rendu de consultation de Chirurgie visc...,K644 K644 K644
284,98354751078,Compte rendu opératoire de Chirurgie viscérale...,L059 L059 L059
285,98956046499,Compte rendu d'hospitalisation du service de C...,K810 K810 K810


In [95]:
df_grouped["dp_gold"] = df_grouped["dp_gold"].str.split().str[0]
df_grouped

,note_id,note_text,dp_gold
0,100158637342,Compte rendu de consultation de Chirurgie visc...,K811
1,100759932764,Compte rendu de consultation de Chirurgie visc...,K801
2,10136122818,Compte rendu d'hospitalisation du service de C...,M205
3,101361228185,Compte rendu opératoire de Chirurgie viscérale...,K429
4,101962523607,Compte rendu de consultation de Chirurgie visc...,K801
...,...,...,...
282,97152160235,Compte rendu d'hospitalisation du service de C...,K602
283,97753455656,Compte rendu de consultation de Chirurgie visc...,K644
284,98354751078,Compte rendu opératoire de Chirurgie viscérale...,L059
285,98956046499,Compte rendu d'hospitalisation du service de C...,K810


In [96]:
df_grouped["dp"] = df_grouped["dp_gold"].apply(lambda x: nlp(x)._.dp)

In [97]:
df_grouped

,note_id,note_text,dp_gold,dp
0,100158637342,Compte rendu de consultation de Chirurgie visc...,K811,C185
1,100759932764,Compte rendu de consultation de Chirurgie visc...,K801,C223
2,10136122818,Compte rendu d'hospitalisation du service de C...,M205,C340
3,101361228185,Compte rendu opératoire de Chirurgie viscérale...,K429,C229
4,101962523607,Compte rendu de consultation de Chirurgie visc...,K801,C223
...,...,...,...,...
282,97152160235,Compte rendu d'hospitalisation du service de C...,K602,E701
283,97753455656,Compte rendu de consultation de Chirurgie visc...,K644,E701
284,98354751078,Compte rendu opératoire de Chirurgie viscérale...,L059,J358
285,98956046499,Compte rendu d'hospitalisation du service de C...,K810,C831


In [98]:
label_attr= ["dp"]
val_data = (
  edsnlp.data.from_pandas(
      df_grouped,
      tokenizer=nlp.tokenizer,
      converter="omop",

  )
)

val_data = val_data.map_pipeline(
    nlp,
)
val_data = val_data.set_processing(num_cpu_workers=8, show_progress=True)

note_nlp = val_data.to_pandas(
    converter="omop",
    doc_attributes=[
        "dp_gold",
        "dp",
    ]
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (766 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (690 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (986 > 512). Running this sequence through the model will result in indexing errors
0it [00:00, ?it/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1021 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (811 > 512). Running this sequence through the model will result in indexing err

In [99]:
note_nlp

,note_id,note_text,dp,entities
0,100158637342,Compte rendu de consultation de Chirurgie visc...,K800,[]
1,100759932764,Compte rendu de consultation de Chirurgie visc...,K801,[]
2,10136122818,Compte rendu d'hospitalisation du service de C...,M2157,[]
3,101361228185,Compte rendu opératoire de Chirurgie viscérale...,K429,[]
4,101962523607,Compte rendu de consultation de Chirurgie visc...,K801,[]
...,...,...,...,...
282,97152160235,Compte rendu d'hospitalisation du service de C...,K602,[]
283,97753455656,Compte rendu de consultation de Chirurgie visc...,K649,[]
284,98354751078,Compte rendu opératoire de Chirurgie viscérale...,L059,[]
285,98956046499,Compte rendu d'hospitalisation du service de C...,K811,[]


In [100]:
note_nlp["dp_gold"] = df_grouped["dp_gold"]

In [101]:
note_nlp

,note_id,note_text,dp,entities,dp_gold
0,100158637342,Compte rendu de consultation de Chirurgie visc...,K800,[],K811
1,100759932764,Compte rendu de consultation de Chirurgie visc...,K801,[],K801
2,10136122818,Compte rendu d'hospitalisation du service de C...,M2157,[],M205
3,101361228185,Compte rendu opératoire de Chirurgie viscérale...,K429,[],K429
4,101962523607,Compte rendu de consultation de Chirurgie visc...,K801,[],K801
...,...,...,...,...,...
282,97152160235,Compte rendu d'hospitalisation du service de C...,K602,[],K602
283,97753455656,Compte rendu de consultation de Chirurgie visc...,K649,[],K644
284,98354751078,Compte rendu opératoire de Chirurgie viscérale...,L059,[],L059
285,98956046499,Compte rendu d'hospitalisation du service de C...,K811,[],K810


In [102]:
compute_metric(note_nlp)

Micro - Precision: 0.3972, Recall: 0.3972, F1: 0.3972
Macro - Precision: 0.1892, Recall: 0.1869, F1: 0.1754


In [103]:
compute_metric(note_nlp, granularity=3)

Micro - Precision: 0.6063, Recall: 0.6063, F1: 0.6063
Macro - Precision: 0.3116, Recall: 0.3153, F1: 0.2975
